# AI Job Agent — PHASE 3: Embedding Retrieval

**Objective:** Take the `candidate_profile` from Phase 2 and `job_embeddings.npy` from Phase 1, and retrieve the top 100–200 jobs by semantic similarity (cosine similarity) before any strict filtering (Phase 4) or detailed matching (Phase 5).

**No LLM calls are used in this notebook.** All vectors are already generated in the previous two phases, so this phase consists purely of matrix multiplication, making it fast and inexpensive.

```text
job_embeddings.npy (Phase 1)      candidate_embedding.npy (Phase 2)
         │                                    │
         └────────────── dot product ─────────┘
                            │
                (Both vectors are unit-norm,
                 so dot product = cosine similarity)
                            │
                  Sort descending + take top K
                            │
                 retrieved_jobs
              (job_id, rank, similarity)
```

## Two Paths, Following the Same Logic as Phase 2

| Path          | Input                                               | Output                                                                       |
| ------------- | --------------------------------------------------- | ---------------------------------------------------------------------------- |
| **B — Live**  | One candidate (`candidate_profile.json`)            | Top K jobs for that candidate, displayed in detail                           |
| **A — Batch** | Multiple candidates (`candidates_prepared.parquet`) | Retrieval for each candidate, serving as direct input for Phase 9 evaluation |

> **Critical requirement:** The `candidate_embedding` must have been generated using the **same model** as the `job_embeddings`. The validation step in §3 enforces this automatically and stops execution if the models do not match.


---
## 1. Setup

In [ ]:
import json
import shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)

WORK = Path("work")
OUT = Path("outputs")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

RUN_ID: 20260917_0857


## 2. Load Phase 1 Outputs (Jobs)

Upload the following four files from the Phase 1 outputs (they were previously stored in the `outputs/` folder, or in `phase1_outputs/` if you downloaded them as a ZIP):

* `jobs_prepared.parquet`
* `job_embeddings.npy`
* `job_ids.npy`
* `run_metadata.json` **(optional, but recommended — used to verify embedding model compatibility)**


In [ ]:
from google.colab import files as _colab_files

print("Upload the Phase 1 files (jobs_prepared.parquet, job_embeddings.npy, "
      "job_ids.npy, run_metadata.json):")
uploaded_p1 = _colab_files.upload()

P1_DIR = WORK / "phase1"
P1_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p1.items():
    (P1_DIR / name).write_bytes(content)

print("\nSaved:")
for f in sorted(P1_DIR.iterdir()):
    print(" -", f.name)

ارفع ملفات Phase 1 (jobs_prepared.parquet, job_embeddings.npy, job_ids.npy, run_metadata.json):


Saving job_embeddings.npy to job_embeddings.npy
Saving job_ids.npy to job_ids.npy
Saving jobs_prepared.parquet to jobs_prepared.parquet
Saving skill_vocabulary.parquet to skill_vocabulary.parquet

Saved:
 - job_embeddings.npy
 - job_ids.npy
 - jobs_prepared.parquet
 - skill_vocabulary.parquet


In [ ]:
jobs_df = pd.read_parquet(P1_DIR / "jobs_prepared.parquet")
job_vectors = np.load(P1_DIR / "job_embeddings.npy")
job_ids = np.load(P1_DIR / "job_ids.npy", allow_pickle=True)

p1_meta_path = P1_DIR / "run_metadata.json"
p1_meta = json.loads(p1_meta_path.read_text()) if p1_meta_path.exists() else None

assert len(jobs_df) == len(job_vectors) == len(job_ids), "row count mismatch across Phase 1 files"
assert (jobs_df["job_id"].to_numpy() == job_ids).all(), "job_id order mismatch — re-export from Phase 1"

print(f"Jobs        : {len(jobs_df):,}")
print(f"Embeddings  : {job_vectors.shape}")
if p1_meta:
    print(f"Embed model : {p1_meta.get('embedding_model')}")
    print(f"Phase 1 run : {p1_meta.get('run_id')}  ({p1_meta.get('created_utc')})")
else:
    print("[NOTE] run_metadata.json not provided — skipping embedding-model cross-check in §4.")

Jobs        : 9,619
Embeddings  : (9619, 1536)
[NOTE] run_metadata.json not provided — skipping embedding-model cross-check in §4.


## 3. Load Phase 2 Outputs (Candidate)

Choose the path based on what you ran in Phase 2:

* **PATH B (Single Candidate):** `candidate_profile.json` + `candidate_embedding.npy`
* **PATH A (Batch):** `candidates_prepared.parquet` + `candidate_embeddings.npy` + `candidate_ids.npy`

Upload either group, or both if you want to test both paths in the same run.


In [ ]:
print("Upload the Phase 2 files (candidate_profile.json + candidate_embedding.npy "
      "و/أو candidates_prepared.parquet + candidate_embeddings.npy + candidate_ids.npy):")
uploaded_p2 = _colab_files.upload()

P2_DIR = WORK / "phase2"
P2_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p2.items():
    (P2_DIR / name).write_bytes(content)

has_live = (P2_DIR / "candidate_profile.json").exists() and (P2_DIR / "candidate_embedding.npy").exists()
has_batch = (P2_DIR / "candidates_prepared.parquet").exists() and (P2_DIR / "candidate_embeddings.npy").exists()

print(f"\nPATH B (live)  available: {has_live}")
print(f"PATH A (batch) available: {has_batch}")
assert has_live or has_batch, "No valid file group was uploaded. Check the file names."

ارفع ملفات Phase 2 (candidate_profile.json + candidate_embedding.npy و/أو candidates_prepared.parquet + candidate_embeddings.npy + candidate_ids.npy):


Saving candidate_embedding.npy to candidate_embedding.npy
Saving candidate_profile.json to candidate_profile.json

PATH B (live)  available: True
PATH A (batch) available: False


---

## 4. Consistency Checks

Before performing any matrix multiplication, verify that the candidate and job embeddings are actually **compatible for comparison**:

* They have the same dimensionality.
* They are unit-norm vectors.
* They were generated using the same embedding model, when `run_metadata.json` is available.

Skipping these checks can silently produce meaningless similarity scores, which is one of the most dangerous silent failure modes in retrieval systems.


In [ ]:
def check_vectors(vectors: np.ndarray, label: str):
    assert vectors.ndim == 2, f"{label}: expected 2D array, got shape {vectors.shape}"
    assert vectors.shape[1] == job_vectors.shape[1], (
        f"{label}: dimension {vectors.shape[1]} != jobs dimension {job_vectors.shape[1]}. "
        f"Candidate and job embeddings were built with different models — "
        f"re-check CONFIG['embedding_model'] in both Phase 1 and Phase 2."
    )
    norms = np.linalg.norm(vectors, axis=1)
    if not np.allclose(norms, 1.0, atol=1e-2):
        bad = int((~np.isclose(norms, 1.0, atol=1e-2)).sum())
        print(f"  [WARNING] {label}: {bad}/{len(vectors)} vectors are not unit-norm "
              f"(min={norms.min():.4f}, max={norms.max():.4f}). "
              f"Re-normalizing automatically — cosine similarity still valid.")
        vectors = vectors / norms[:, None].clip(min=1e-9)
    return vectors


job_vectors = check_vectors(job_vectors, "job_embeddings")
print(f"job_embeddings OK — shape {job_vectors.shape}")

if p1_meta and p1_meta.get("embedding_model"):
    p1_model = p1_meta["embedding_model"]
    print(f"\nPhase 1 embedding model: {p1_model}")
    print("Make sure Phase 2's CONFIG['embedding_model'] matched this exactly — "
          "there is no automatic cross-notebook check possible beyond the "
          "dimension check above, since candidate_profile.json does not "
          "currently store which model built its embedding.")

job_embeddings OK — shape (9619, 1536)


In [ ]:
live_vector = None
live_profile = None
batch_df = None
batch_vectors = None
batch_ids = None

if has_live:
    live_profile = json.loads((P2_DIR / "candidate_profile.json").read_text())
    live_vector = np.load(P2_DIR / "candidate_embedding.npy")
    live_vector = check_vectors(live_vector.reshape(1, -1), "candidate_embedding (live)")[0]
    print(f"Live candidate : {live_profile.get('most_recent_title')} "
          f"({live_profile['candidate_id']})")

if has_batch:
    batch_df = pd.read_parquet(P2_DIR / "candidates_prepared.parquet")
    batch_vectors = np.load(P2_DIR / "candidate_embeddings.npy")
    batch_ids = np.load(P2_DIR / "candidate_ids.npy", allow_pickle=True)

    assert len(batch_df) == len(batch_vectors) == len(batch_ids), \
        "row count mismatch across Phase 2 batch files"
    assert (batch_df["candidate_id"].to_numpy() == batch_ids).all(), \
        "candidate_id order mismatch — re-export from Phase 2"

    batch_vectors = check_vectors(batch_vectors, "candidate_embeddings (batch)")
    print(f"Batch candidates: {len(batch_df)}")

Live candidate : Freelance AI Engineer (e07ca2443daccbb5)


---
## 5. Configuration

In [ ]:
CONFIG = {
    # Number of jobs retrieved per candidate — feeds into Phase 4 (Hard Filtering) later
    "top_k": 150,

    # Optional minimum similarity threshold (0-1). None = no threshold; take the top_k jobs regardless of their score.
    "min_similarity": None,
}

print("CONFIG:", CONFIG)

CONFIG: {'top_k': 150, 'min_similarity': None}


---

## 6. Core Retrieval Function

Both `job_vectors` and `candidate_vector` are unit-norm vectors (their length is 1), so the **dot product is directly equivalent to cosine similarity**, with no additional division by vector lengths.

We use `argpartition` instead of fully sorting the array with `argsort` to retrieve the top K elements in **O(n)** time rather than **O(n log n)**. This can make a significant difference when working with a large corpus.


In [ ]:
from typing import Optional


def retrieve_top_k(query_vector: np.ndarray,
                    matrix: np.ndarray,
                    k: int,
                    min_similarity: Optional[float] = None) -> tuple:
    """Return (indices, similarities) of the top-k rows in `matrix` most
    similar to `query_vector`, sorted descending. Assumes both are unit-norm."""

    sims = matrix @ query_vector

    if min_similarity is not None:
        eligible = np.where(sims >= min_similarity)[0]
        if len(eligible) == 0:
            return np.array([], dtype=int), np.array([], dtype=np.float32)
        sims_eligible = sims[eligible]
        k_eff = min(k, len(eligible))
        part = np.argpartition(-sims_eligible, k_eff - 1)[:k_eff]
        order = np.argsort(-sims_eligible[part])
        top_local = part[order]
        return eligible[top_local], sims_eligible[top_local]

    k_eff = min(k, len(sims))
    part = np.argpartition(-sims, k_eff - 1)[:k_eff]
    order = np.argsort(-sims[part])
    top_idx = part[order]
    return top_idx, sims[top_idx]


# Quick sanity check with known values before using it on real data
_test_matrix = np.eye(5, dtype=np.float32)
_test_query = np.array([0, 0, 1, 0, 0], dtype=np.float32)
_idx, _sim = retrieve_top_k(_test_query, _test_matrix, k=3)
assert _idx[0] == 2 and abs(_sim[0] - 1.0) < 1e-6, "retrieve_top_k sanity check failed"
print("retrieve_top_k sanity check passed.")

retrieve_top_k sanity check passed.


---
## 7. PATH B — Live candidate retrieval

In [ ]:
assert live_vector is not None, (
  "No single candidate is loaded. Upload candidate_profile.json + candidate_embedding.npy "
"in §3, or proceed to §9 (PATH A) if you are working with the batch path only."
)

top_idx, top_sims = retrieve_top_k(
    live_vector, job_vectors, CONFIG["top_k"], CONFIG["min_similarity"]
)

live_results = jobs_df.iloc[top_idx].copy()
live_results.insert(0, "rank", range(1, len(live_results) + 1))
live_results.insert(1, "similarity", top_sims)

print(f"Retrieved {len(live_results)} jobs for candidate "
      f"{live_profile.get('most_recent_title', live_profile['candidate_id'])}")
print(f"Similarity range: {top_sims.min():.4f} – {top_sims.max():.4f}")

Retrieved 150 jobs for candidate Freelance AI Engineer
Similarity range: 0.5575 – 0.6697


### 7.1 Review the top matches

In [ ]:
display_cols = ["rank", "similarity", "title_clean", "company_clean",
                "city", "country", "experience_level", "work_arrangement"]
display_cols = [c for c in display_cols if c in live_results.columns]

live_results[display_cols].head(20).style.format({"similarity": "{:.4f}"})

,rank,similarity,title_clean,company_clean,city,country,experience_level,work_arrangement
4668,1,0.6697,AI/ML Data Engineer,randstad.in,Perungudi,India,Not Specified,Not Specified
3603,2,0.6583,Senior/Lead Data Scientist - Agentic AI,Tiger Analytics Inc.,San Francisco,United States,Senior Level,Not Specified
4047,3,0.6552,AI Data Engineer,Envision Technology Solutions,None,United States,Senior Level,Remote
2362,4,0.6528,AI Engineer,CodeNinja,Dubai,UAE,Mid Level,Not Specified
4897,5,0.6433,Senior/Lead Data Scientist - Agentic AI,Tiger Analytics Inc.,New York,United States,Senior Level,Not Specified
8116,6,0.6400,"Senior AI/ML Engineer (Python,RAG & Multimodal AI)",Apptad Inc,Montreal,None,Senior Level,On-site
2901,7,0.6380,AI Engineer,Mobiik,Dubai,UAE,Not Specified,Not Specified
5636,8,0.6325,AI Research Scientist,Openkyber,Alaska,United States,Senior Level,On-site
7175,9,0.6325,Data Scientist - NLP,XA Group,Noida,India,Not Specified,Not Specified
6449,10,0.6121,Data Scientist II,Robert Half,United State,United States,Senior Level,Not Specified


### 7.2 Diagnostics

The distribution of similarity scores can reveal potential issues early:

* If the top 150 results all have similarity scores above **0.85**, the corpus may be too narrowly concentrated around the candidate’s profile. This is not necessarily a problem, but it is worth investigating.
* If the top result has a similarity score below **0.3**, the Phase 1 corpus likely does not adequately cover the candidate’s domain. Review the **anchors** configured in `CONFIG`.


In [ ]:
print(pd.Series(top_sims).describe().round(4).to_string())

if top_sims.max() < 0.30:
    print(
        "\n[WARNING] Highest similarity is below 0.30 — the job corpus "
        "likely does not cover this candidate's domain. Review the "
        "anchors in the Phase 1 CONFIG or add a new anchor covering "
        "this domain."
    )

elif (top_sims > 0.85).mean() > 0.5:
    print(
        "\n[NOTE] More than half of the results have similarity above "
        "0.85 — the corpus may be narrowly focused on a single specialty. "
        "This is acceptable if intentional."
    )

else:
    print("\nThe similarity distribution looks normal.")

count    150.0000
mean       0.5804
std        0.0224
min        0.5575
25%        0.5640
50%        0.5733
75%        0.5915
max        0.6697

توزيع التشابه يبدو طبيعيًا.


### 7.3 Save PATH B output

In [ ]:
LIVE_OUT = OUT / "live_candidate"
LIVE_OUT.mkdir(parents=True, exist_ok=True)

retrieval_cols = ["rank", "similarity", "job_id"] + [
    c for c in jobs_df.columns if c not in ("job_id",)
]
retrieval_cols = [c for c in retrieval_cols if c in live_results.columns]

live_results[retrieval_cols].to_parquet(
    LIVE_OUT / "retrieved_jobs.parquet", index=False
)

print(f"Saved {LIVE_OUT / 'retrieved_jobs.parquet'}  ({len(live_results)} rows)")

Saved outputs/live_candidate/retrieved_jobs.parquet  (150 rows)


---

## 8. PATH A — Batch Retrieval (Evaluation Set)

Retrieve the top K jobs **for each candidate** in `candidates_prepared.parquet`.

This serves as the direct input for building the Phase 9 evaluation set, including **Precision, Recall, and NDCG** metrics later.

> If you are working with the single-candidate path only, skip this entire section.


In [ ]:
# assert batch_df is not None, (
#     "No candidate batch is loaded. Upload candidates_prepared.parquet + "
#     "candidate_embeddings.npy + candidate_ids.npy in §3, or skip this section "
#     "if you are working with PATH B only."
# )

# all_rows = []

# for i in range(len(batch_df)):
#     cand_id = batch_ids[i]
#     cand_vector = batch_vectors[i]

#     idx, sims = retrieve_top_k(
#         cand_vector,
#         job_vectors,
#         CONFIG["top_k"],
#         CONFIG["min_similarity"]
#     )

#     for rank, (j, s) in enumerate(zip(idx, sims), start=1):
#         all_rows.append({
#             "candidate_id": cand_id,
#             "job_id": job_ids[j],
#             "rank": rank,
#             "similarity": float(s),
#         })

# batch_retrieval_df = pd.DataFrame(all_rows)

# print(
#     f"Retrieved {len(batch_retrieval_df):,} (candidate, job) pairs "
#     f"across {len(batch_df)} candidates "
#     f"(~{len(batch_retrieval_df) / max(len(batch_df), 1):.0f} per candidate)"
# )

AssertionError: لا توجد مجموعة مرشحين مُحمَّلة. ارفع candidates_prepared.parquet + candidate_embeddings.npy + candidate_ids.npy في §3، أو تخطَّ هذا القسم إذا كنت تعمل بمسار PATH B فقط.

### 8.1 Per-candidate summary

In [ ]:
# summary = (
#     batch_retrieval_df.groupby("candidate_id")["similarity"]
#     .agg(["count", "max", "mean", "min"])
#     .round(4)
#     .reset_index()
# )

# # Link candidate names/titles for readability, if available
# name_col = "most_recent_title" if "most_recent_title" in batch_df.columns else "candidate_id"
# summary = summary.merge(
#     batch_df[["candidate_id", name_col]], on="candidate_id", how="left"
# )

# low_coverage = summary[summary["max"] < 0.30]
# if len(low_coverage):
#     print(
#         "[WARNING] These candidates have a highest similarity score below 0.30 — "
#         "the job corpus likely does not cover their domain:"
#     )
#     print(low_coverage[["candidate_id", name_col, "max"]].to_string(index=False))
#     print()

# summary[["candidate_id", name_col, "count", "max", "mean", "min"]]

### 8.2 Save PATH A output

In [ ]:
# batch_retrieval_path = OUT / "retrieved_jobs_batch.parquet"
# batch_retrieval_df.to_parquet(batch_retrieval_path, index=False)

# print(f"Saved {batch_retrieval_path}  ({len(batch_retrieval_df):,} rows, "
#       f"{len(batch_df)} candidates × up to {CONFIG['top_k']} jobs)")

---
## 9. Data dictionary

In [ ]:
DATA_DICTIONARY = """# Phase 3 outputs — Data Dictionary

## live_candidate/retrieved_jobs.parquet (PATH B)

Top-K jobs for a single candidate, ranked by cosine similarity. Full job
columns from jobs_prepared.parquet are included (joined), plus:

| Column | Notes |
|---|---|
| rank | 1 = most similar. Contiguous, no gaps. |
| similarity | Cosine similarity in [-1, 1] (in practice ~[0, 1] for this domain). |
| job_id | Foreign key into jobs_prepared.parquet from Phase 1. |

This file is the direct input to PHASE 4 (Hard Filtering) — filter its rows
on country/city/work_arrangement/etc. rather than re-querying job_embeddings.

## retrieved_jobs_batch.parquet (PATH A)

Same shape, long format, one row per (candidate, job, rank) triple:
`candidate_id, job_id, rank, similarity`. Deliberately lean (no job detail
columns) since it covers many candidates × top_k jobs each — join against
jobs_prepared.parquet on `job_id` when job details are needed.

This is the direct input to PHASE 9 (Evaluation): for each candidate_id,
compare the retrieved job_id set (and their ranks) against a manually
labeled ground-truth set of relevant jobs to compute Precision@K, Recall@K,
and NDCG@K.

## Retrieval contract
- Similarity = dot product of two L2-normalized (unit-norm) vectors = cosine similarity.
- CONFIG["top_k"] controls how many jobs are kept per candidate (default 150).
- CONFIG["min_similarity"], if set, drops jobs below that threshold even if
  fewer than top_k remain — useful to avoid feeding Phase 4/5 with jobs that
  are not meaningfully related to the candidate at all.
- No re-ranking happens here — this is pure embedding-space retrieval.
  Re-ranking by a stronger signal (LLM or cross-encoder) is PHASE 6.
"""

(OUT / "PHASE3_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

---
## 10. Final verification

In [ ]:
print("=" * 60)
print("PATH B (live candidate)")
print("=" * 60)

live_path = OUT / "live_candidate" / "retrieved_jobs.parquet"
if live_path.exists():
    reloaded = pd.read_parquet(live_path)
    assert reloaded["rank"].min() == 1
    assert reloaded["rank"].is_monotonic_increasing
    assert reloaded["similarity"].is_monotonic_decreasing
    assert reloaded["job_id"].is_unique
    print(f"  rows       : {len(reloaded)}")
    print(f"  top match  : {reloaded.iloc[0]['title_clean']} "
          f"(sim={reloaded.iloc[0]['similarity']:.4f})")
    print("  STATUS: ready for Phase 4")
else:
    print("  Not run — skip if you only used PATH A.")

print()
print("=" * 60)
print("PATH A (batch)")
print("=" * 60)

if (OUT / "retrieved_jobs_batch.parquet").exists():
    reloaded_b = pd.read_parquet(OUT / "retrieved_jobs_batch.parquet")
    assert reloaded_b.groupby("candidate_id")["job_id"].apply(lambda s: s.is_unique).all()
    print(f"  rows       : {len(reloaded_b):,}")
    print(f"  candidates : {reloaded_b['candidate_id'].nunique()}")
    print("  STATUS: ready for Phase 9 evaluation set")
else:
    print("  Not run — skip if you only used PATH B.")

---

## 11. Persist Outputs

Direct download is the simplest and most reliable option compared with `drive.mount()`, which can sometimes fail with a `credential propagation` error unrelated to the code.

If Drive works for you, add a `drive.mount()` cell along with `shutil.copy2`, as in Phase 1/2, if desired.


In [ ]:
zip_path = shutil.make_archive("phase3_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")

_colab_files.download(zip_path)

Created /content/phase3_outputs.zip (0.43 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Next Step — PHASE 4

`retrieved_jobs.parquet` (or `retrieved_jobs_batch.parquet`) contains all the original job columns from Phase 1. Now we can apply simple, strict filtering based on:

`country`, `city`, `work_arrangement` (only where `work_arrangement_inferred == False`), `education_level`, and `nationality_requirement`.

There is no need to reload anything. All the required fields are already available in this file.
